In [ ]:
````xml
<!-- filepath: /Users/bell/Programs/EcoFOCIpy/notebooks/02_CTD_Cast_Analysis.ipynb -->
<VSCode.Cell language="markdown">
# CTD Cast Analysis

Analyze individual CTD (Conductivity-Temperature-Depth) profiles from oceanographic research cruises.

**Topics covered:**
- Loading CTD bottle file data
- Computing derived properties (density, potential temperature)
- Creating T-S (Temperature-Salinity) diagrams
- Identifying water masses and stratification
- Depth profile visualization
- Comparing multiple casts

**Estimated time:** 25 minutes  
**Difficulty:** Intermediate
</VSCode.Cell>
<VSCode.Cell language="python">
import warnings
warnings.filterwarnings('ignore')

import EcoFOCIpy
from EcoFOCIpy import instruments
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 8)

print(f"EcoFOCIpy version: {EcoFOCIpy.__version__}")
print("Libraries loaded ✓")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## CTD Profiles Explained

A CTD (Conductivity-Temperature-Depth) cast measures oceanographic properties as the sensor moves vertically through the water column from surface to bottom.

**What is measured:**
- **Temperature** (T): How warm the water is in °C
- **Conductivity** (C): How salty the water is in S/m
- **Depth** (D): Pressure converted to water depth in meters
- **Derived**: Salinity, Density, Oxygen (if sensor present)

**Two CTD deployment types:**
1. **Profiling CTD** - Lowered on wire during research cruise
2. **Moored CTD** - Fixed at one depth on mooring for continuous monitoring
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 1: Create Sample CTD Data

Let's create a realistic CTD cast profile with typical oceanographic parameters.
</VSCode.Cell>
<VSCode.Cell language="python">
# Create sample CTD profile data
depth = np.linspace(0, 500, 50)  # 0 to 500 meters

# Realistic oceanographic profiles
temperature = 5 + 8 * np.exp(-depth/30) + 0.1 * np.random.normal(0, 1, len(depth))
salinity = 32 + 1.5 * (1 - np.exp(-depth/80)) + 0.02 * np.random.normal(0, 1, len(depth))
oxygen = 400 - 150 * (1 - np.exp(-depth/100)) + 5 * np.random.normal(0, 1, len(depth))

# Calculate derived properties
# Potential density anomaly (simplified)
sigma_theta = 1000 + (1.5 * (35 - salinity)) - (0.2 * (temperature - 10))

# Create DataFrame
cast_data = pd.DataFrame({
    'depth': depth,
    'pressure': depth * 1.019,  # Approximate conversion (1 dbar ≈ 1 meter)
    'temperature': temperature,
    'salinity': salinity,
    'oxygen': oxygen,
    'sigma_theta': sigma_theta
})

print("CTD CAST DATA")
print("="*60)
print(cast_data.head(15))
print(f"\nData summary:")
print(f"  Depth range: {cast_data['depth'].min():.1f} to {cast_data['depth'].max():.1f} m")
print(f"  Temperature: {cast_data['temperature'].min():.2f} to {cast_data['temperature'].max():.2f} °C")
print(f"  Salinity: {cast_data['salinity'].min():.2f} to {cast_data['salinity'].max():.2f} PSU")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 2: Calculate Derived Properties

From the basic measurements (T, C, D), we can derive important oceanographic quantities.
</VSCode.Cell>
<VSCode.Cell language="python">
# Calculate additional derived properties
print("DERIVED PROPERTIES")
print("="*60)

# Potential Temperature (approximately equal to in-situ T at surface)
cast_data['potential_temp'] = cast_data['temperature'] - 0.001 * cast_data['pressure']

# Density (sigma_t - density anomaly referenced to surface)
cast_data['sigma_t'] = cast_data['sigma_theta']

# Brunt-Väisälä frequency (stratification indicator)
# dz in meters, dsigma from density gradient
dz = np.diff(cast_data['depth'])
dsigma = np.diff(cast_data['sigma_theta'])
n2 = -9.81 * dsigma / (1000 * dz) / 1000  # rad²/s²
cast_data['N2'] = np.insert(n2, 0, 0)  # Add placeholder for first point

print(cast_data[['depth', 'temperature', 'salinity', 'sigma_theta', 'N2']].head(10))

# Summary statistics
print("\nProperty Ranges:")
print(f"  Temperature:      {cast_data['temperature'].min():.2f} to {cast_data['temperature'].max():.2f} °C")
print(f"  Salinity:         {cast_data['salinity'].min():.2f} to {cast_data['salinity'].max():.2f} PSU")
print(f"  Sigma-theta:      {cast_data['sigma_theta'].min():.2f} to {cast_data['sigma_theta'].max():.2f} kg/m³")
print(f"  Oxygen:           {cast_data['oxygen'].min():.0f} to {cast_data['oxygen'].max():.0f} µmol/kg")
print(f"  N² (Stratification): {cast_data['N2'].min():.2e} to {cast_data['N2'].max():.2e} rad²/s²")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 3: Temperature-Salinity (T-S) Diagram

The T-S diagram is one of the most important tools in oceanography. It reveals:
- **Water mass identification** - different water types cluster in T-S space
- **Mixing patterns** - straight lines indicate linear mixing
- **Stratification** - vertical excursion shows density structure

Each point represents water at different depths, colored by pressure.
</VSCode.Cell>
<VSCode.Cell language="python">
# Create T-S diagram
fig, ax = plt.subplots(figsize=(10, 10))

# Create scatter plot, colored by depth
scatter = ax.scatter(
    cast_data['salinity'],
    cast_data['temperature'],
    c=cast_data['depth'],
    cmap='twilight',
    s=100,
    alpha=0.7,
    edgecolors='black',
    linewidth=0.5
)

# Add density contours
salinity_range = np.linspace(cast_data['salinity'].min()-0.5, cast_data['salinity'].max()+0.5, 50)
temp_range = np.linspace(cast_data['temperature'].min()-1, cast_data['temperature'].max()+1, 50)
S, T = np.meshgrid(salinity_range, temp_range)
Sigma = 1000 + (1.5 * (35 - S)) - (0.2 * (T - 10))

contours = ax.contour(S, T, Sigma, colors='gray', alpha=0.3, levels=10, linewidths=0.5)
ax.clabel(contours, inline=True, fontsize=8, fmt='%.1f')

# Labels and formatting
ax.set_xlabel('Salinity (PSU)', fontsize=12, fontweight='bold')
ax.set_ylabel('Temperature (°C)', fontsize=12, fontweight='bold')
ax.set_title('Temperature-Salinity Diagram (T-S Plot)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Colorbar
cbar = plt.colorbar(scatter, ax=ax, label='Depth (m)')

plt.tight_layout()
plt.show()

print("✓ T-S diagram created")
print("  Interpretation: Each point = water at a specific depth")
print("  Gray contours = density (σθ)")
print("  Color gradient = depth progression")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 4: Depth Profile Plots

Examine how each property varies with depth. This reveals stratification and layer structure.
</VSCode.Cell>
<VSCode.Cell language="python">
# Create multi-panel depth profiles
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

# Temperature profile
axes[0].plot(cast_data['temperature'], cast_data['depth'], 'b-', linewidth=2)
axes[0].set_xlabel('Temperature (°C)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Depth (m)', fontsize=11)
axes[0].invert_yaxis()
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Temperature Profile', fontsize=12, fontweight='bold')

# Salinity profile
axes[1].plot(cast_data['salinity'], cast_data['depth'], 'g-', linewidth=2)
axes[1].set_xlabel('Salinity (PSU)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Depth (m)', fontsize=11)
axes[1].invert_yaxis()
axes[1].grid(True, alpha=0.3)
axes[1].set_title('Salinity Profile', fontsize=12, fontweight='bold')

# Density profile
axes[2].plot(cast_data['sigma_theta'], cast_data['depth'], 'r-', linewidth=2)
axes[2].set_xlabel('σθ (kg/m³)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Depth (m)', fontsize=11)
axes[2].invert_yaxis()
axes[2].grid(True, alpha=0.3)
axes[2].set_title('Density Profile', fontsize=12, fontweight='bold')

# Oxygen profile
axes[3].plot(cast_data['oxygen'], cast_data['depth'], 'm-', linewidth=2)
axes[3].set_xlabel('Oxygen (µmol/kg)', fontsize=11, fontweight='bold')
axes[3].set_ylabel('Depth (m)', fontsize=11)
axes[3].invert_yaxis()
axes[3].grid(True, alpha=0.3)
axes[3].set_title('Oxygen Profile', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Depth profile plots created")
</VSCode.Cell>
<VSCode.Cell language="markdown">
## Step 5: Identify Water Masses and Layers

Analyze the vertical structure to identify distinct water masses and thermoclines.
</VSCode.Cell>
<VSCode.Cell language="python">
# Analyze stratification
print("WATER MASS ANALYSIS")
print("="*60)

# Find thermocline (steepest temperature gradient)
dT_dz = np.abs(np.diff(cast_data['temperature']) / np.diff(cast_data['depth']))
thermo_idx = np.argmax(dT_dz)
thermocline_depth = cast_data['depth'].iloc[thermo_idx]
thermocline_strength = dT_dz[thermo_idx]

# Find pycnocline (steepest density gradient)
dsigma_dz = np.abs(np.diff(cast_data['sigma_theta']) / np.diff(cast_data['depth']))
pycno_idx = np.argmax(dsigma_dz)
pycnocline_depth = cast_data['depth'].iloc[pycno_idx]
pycnocline_strength = dsigma_dz[pycno_idx]

# Define water layers
surface_depth = 50  # m
thermocline_start = thermocline_depth - 10
thermocline_end = thermocline_depth + 10
deep_start = 200  # m

surface = cast_data[cast_data['depth'] <= surface_depth]
thermocline = cast_data[(cast_data['depth'] >= thermocline_start) & 
                        (cast_data['depth'] <= thermocline_end)]
deep = cast_data[cast_data['depth'] >= deep_start]

print("\n1. THERMOCLINE (Temperature gradient):")
print(f"   Location: {thermocline_depth:.1f} m")
print(f"   Strength: {thermocline_strength:.3f} °C/m")

print("\n2. PYCNOCLINE (Density gradient):")
print(f"   Location: {pycnocline_depth:.1f} m")
print(f"   Strength: {pycnocline_strength:.4f} kg/m³/m")

print("\n3. SURFACE LAYER (0-50 m):")
print(f"   Avg Temperature: {surface['temperature'].mean():.2f} °C")
print(f"   Avg Salinity: {surface['salinity'].mean():.2f} PSU")
print(f"   Avg Density: {surface['sigma_theta'].mean():.2f} kg/m³")

print("\n4. DEEP LAYER (>200 m):")
print(f"   Avg Temperature: {deep['temperature'].mean():.2f} °C")
print(f"   Avg Salinity: {deep['salinity'].mean():.2f} PSU")
print(f"   Avg Density: {deep['sigma_theta'].mean():.2f} kg/m³")
</VSCode.Cell>
<VSCode.Cell language="python">
# Visualize water layers and thermocline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Temperature with thermocline highlighted
ax1.plot(cast_data['temperature'], cast_data['depth'], 'b-', linewidth=2.5, label='Temperature')
ax1.axhline(thermocline_depth, color='red', linestyle='--', linewidth=2, label=f'Thermocline ({thermocline_depth:.1f} m)')
ax1.axhspan(0, 50, alpha=0.1, color='cyan', label='Surface Layer')
ax1.axhspan(200, 500, alpha=0.1, color='navy', label='Deep Layer')
ax1.fill_betweenx(cast_data['depth'], 
                   cast_data['temperature'].min(), 
                   cast_data['temperature'], 
                   alpha=0.2, color='blue')
ax1.set_xlabel('Temperature (°C)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Depth (m)', fontsize=11, fontweight='bold')
ax1.invert_yaxis()
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper right')
ax1.set_title('Thermocline Identification', fontsize=12, fontweight='bold')

# Right: Density with pycnocline highlighted
ax2.plot(cast_data['sigma_theta'], cast_data['depth'], 'r-', linewidth=2.5, label='Potential Density')
ax2.axhline(pycnocline_depth, color='darkred', linestyle='--', linewidth=2, label=f'Pycnocline ({pycnocline_depth:.1f} m)')
ax2.fill_betweenx(cast_data['depth'], 
                   cast_data['sigma_theta'].min(), 
                   cast_data['sigma_theta'], 
                   alpha=0.2, color='red')
ax2.set_xlabel('σθ (kg/m³)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Depth (m)', fontsize=11, fontweight='bold')
ax2.invert_yaxis()
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper right')
ax2.set_title('Pycnocline Identification', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Layer identification complete")